In [2]:
import numpy as np
import csv
import pandas as pd
from itertools import islice
import json
import missingno as msno
import matplotlib.pyplot as plt

In [3]:
books = pd.read_csv('../data/final_book_dataset.csv', delimiter='\t')

/var/folders/bt/8s6v7ngs6m93f2x58bpj1zl00000gn/T/ipykernel_63969/2359748259.py:1: DtypeWarning: Columns (0: isbn) have mixed types. Specify dtype option on import or set low_memory=False.
  books = pd.read_csv('../data/final_book_dataset.csv', delimiter='\t')


In [4]:
books[books.duplicated(['title', 'author'], keep=False)].sort_values(by='title')

,Unnamed: 0,title_id,title,author,release_year,release_date,first_publisher,author_age_at_release,author_birthplace,isbn,book_synopsis,tags,hugo,locus


In [5]:
books.head(5)

,Unnamed: 0,title_id,title,author,release_year,release_date,first_publisher,author_age_at_release,author_birthplace,isbn,book_synopsis,tags,hugo,locus
0,7649,9372.0,The Long Loud Silence,Wilson Tucker,1954,1954-00-00,Dell,40.0,"Deer Creek, Illinois, USA",0899683754,Publisher's description: Tomorrow's war -- the...,"['Anatomy of Wonder 1 Core Collection', 'biolo...",False,False
1,82380,2215181.0,Mrs. Candy Strikes It Rich,Robert Tallant,1954,1954-00-00,Doubleday,45.0,"New Orleans, Louisiana, USA",NaN,NaN,NaN,False,False
2,53874,1392436.0,Return to the Lost Planet,Angus MacVicar,1954,1954-00-00,Burke,46.0,"Duror, Argyll, Scotland, UK",NaN,NaN,NaN,False,False
3,41661,1112206.0,Rainbow on the Road,Esther Forbes,1954,1954-00-00,Houghton Mifflin,63.0,"Westborough, Massachusetts, USA",NaN,NaN,NaN,False,False
4,1728,1908.0,The Forgotten Planet,Murray Leinster,1954,1954-00-00,Ace Books,58.0,"Norfolk, Virginia, USA",0881846163,<b>From the first page of the Ace Double:</b> ...,"['insects', 'Librivox', 'Project Gutenberg', '...",False,False


In [6]:
finalists = books[books['hugo'] | books['locus']]
nonfinalists = books[~(books['hugo'] | books['locus'])]

### Publisher Info

In [7]:
finalists['first_publisher'].value_counts().sort_index()

first_publisher
47North                             2
Ace Books                         136
Ace Books / SFBC                    4
Ace Fantasy Books                   3
Ace Science Fiction Books           5
                                 ... 
William Morrow / HarperCollins     13
William Morrow / QPBC               1
William Morrow / SFBC               1
Windrush                            1
ibooks                              1
Name: count, Length: 336, dtype: int64

Publisher will be a useful feature, but currently many publishers have slightly different names!

In [8]:
def clean_publishers(publisher):
    # "/" separates an imprint/collection from its parent company, keep only imprint/collection:
    pub = str(publisher).split("/")[0].strip()

    # turn to lowercase
    pub = pub.lower()

    common_publishers = ['tor', 'ace', 'del rey', 'doubleday', 'bloomsbury', 'orbit', 'gollancz', 'daw', 'harpercollins', 'macmillan', 'simon & schuster']
    for cp in common_publishers:
        if cp in pub:
            # print(cp, pub)
            pub = cp

    # remove bracketed items like (us) (uk)
    pub = pub.split("(")[0].strip()

    # # remove some filler words that cause differences between same publishers
    # removals = [" science fiction", ' fantasy', ' uk', ' us', '(uk)', '(us)','.com', ' press']
    # for remove in removals:
    #     pub = pub.replace(remove, '')
    return pub.replace(' ', '')

In [9]:
finalists['first_publisher'] = finalists['first_publisher'].apply(clean_publishers)
nonfinalists['first_publisher'] = nonfinalists['first_publisher'].apply(clean_publishers)

# finalists2.value_counts().sort_index()

In [10]:
finalists['first_publisher'].value_counts(normalize=True)#.reset_index(name='counts').set_index('first_publisher').plot.pie(y='counts', legend=False)

first_publisher
tor                                          0.191655
ace                                          0.074721
delrey                                       0.055798
bantamspectra                                0.051431
orbit                                        0.045609
                                               ...   
rebellion                                    0.000485
recordedbooks,inc.andblackstonepublishing    0.000485
puffin                                       0.000485
clashbooks                                   0.000485
williammorrow&harperaudio                    0.000485
Name: proportion, Length: 262, dtype: float64

In [11]:
nonfinalists['first_publisher'].value_counts(normalize=True)#.reset_index(name='counts').set_index('first_publisher').plot.pie(y='counts', legend=False)

first_publisher
ace                      0.027459
tor                      0.023413
lmbpnpublishing          0.008955
delrey                   0.007905
pocketbooks              0.007871
                           ...   
openroadentertainment    0.000007
daphnepress              0.000007
lovisebooks              0.000007
heliumbeach              0.000007
baskerville              0.000007
Name: proportion, Length: 15427, dtype: float64

### Authors

In [103]:
finalists['author'].value_counts()

author
C. J. Cherryh         41
Gene Wolfe            26
Poul Anderson         20
Robert Silverberg     20
Joe Haldeman          20
                      ..
Sarah Rees Brennan     1
Claire North           1
R. F. Kuang            1
Antonia Hodgson        1
Isaac Fellman          1
Name: count, Length: 588, dtype: int64

In [104]:
nonfinalists['author'].value_counts()

author
R. L. Stine               329
Michael Anderle           281
Eve Langlais              202
Odette C. Bell            202
D. K. Holmberg            187
                         ... 
West Ambrose                1
Madeline Bell               1
Leslie Adame                1
Maddie Martinez             1
Samantha Browning Shea      1
Name: count, Length: 39993, dtype: int64

### Tags

In [92]:
all_finalist_tags = [[item.strip().strip("'").lower() for item in taglist.strip('[]').split(',')]
    for taglist in finalists['tags'].tolist()
    if isinstance(taglist, str)]

flat_list_f = pd.Series([tag for taglist in all_finalist_tags for tag in taglist])

all_nf_tags = [[item.strip().strip("'").lower() for item in taglist.strip('[]').split(',')]
    for taglist in nonfinalists['tags'].tolist()
    if isinstance(taglist, str)]

flat_list_nf = pd.Series([tag for taglist in all_nf_tags for tag in taglist])

In [93]:
flat_list_f.value_counts()

science fiction     2226
fantasy             1776
fiction             1150
adventure            710
adventurous          522
                    ... 
liminal                1
thoughtful             1
healing                1
single pov             1
first person pov       1
Name: count, Length: 2873, dtype: int64

In [94]:
flat_list_nf.value_counts()

fantasy                       36468
science fiction               23551
fiction                       19613
adventure                     11512
young adult                    9993
                              ...  
gross                             1
aardvark                          1
fiction / horror / general        1
irish folk                        1
rover                             1
Name: count, Length: 8451, dtype: int64

In [96]:
flat_list2_f = flat_list_f[~flat_list_f.str.contains('award')]
# remove all the ones which say they won an award...

flat_list2_nf = flat_list_nf[~flat_list_nf.str.contains('award')]


In [97]:
# flat_list2 = flat_list2[~((flat_list2 == 'fantasy') | (flat_list2 == 'fiction') | (flat_list2 == 'science fiction'))]


In [98]:
flat_list2_f.value_counts()

science fiction     2226
fantasy             1776
fiction             1150
adventure            710
adventurous          522
                    ... 
liminal                1
thoughtful             1
healing                1
single pov             1
first person pov       1
Name: count, Length: 2819, dtype: int64

In [99]:
flat_list2_nf.value_counts()

fantasy                       36468
science fiction               23551
fiction                       19613
adventure                     11512
young adult                    9993
                              ...  
gross                             1
aardvark                          1
fiction / horror / general        1
irish folk                        1
rover                             1
Name: count, Length: 8399, dtype: int64